# 01 · Explore the central-complex graph
The cell-type-level connectivity exported from MaleCNS v1.0 by `scripts/fetch_connectome.py`. Each entry is the
mean number of synapses per (pre-cell, post-cell) pair between two cell types, signed by the presynaptic type's
predicted neurotransmitter (ACh +, GABA/Glu −).

In [ ]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from flybrain_worldle.connectome.central_complex import CXGraph
g = CXGraph.load()
print(f"{g.n_types} cell types, {int(g.n_cells.sum())} cells, {(g.weights != 0).sum()} type->type edges")
print(f"signed types: +{(g.signs > 0).sum()}  -{(g.signs < 0).sum()}  unknown {(g.signs == 0).sum()}")

In [ ]:
order = np.argsort(-g.n_cells)
w = g.weights[np.ix_(order, order)]
fig, ax = plt.subplots(figsize=(8, 8))
lim = np.percentile(np.abs(w[w != 0]), 98)
ax.imshow(np.sign(w) * np.log1p(np.abs(w)) / np.log1p(lim), cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_title("central-complex type-to-type connectivity (rows: pre, cols: post)")
ax.set_xticks([]); ax.set_yticks([]); plt.show()

In [ ]:
deg_out = (g.weights != 0).sum(1); deg_in = (g.weights != 0).sum(0)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].hist(deg_out, bins=30); axes[0].set_title("out-degree (types)")
axes[1].hist(deg_in, bins=30); axes[1].set_title("in-degree (types)")
plt.show()
pd.DataFrame({"type": g.types, "cells": g.n_cells, "sign": g.signs, "out": deg_out, "in": deg_in}).sort_values("cells", ascending=False).head(25)

The same graph after `CXGraph.shuffled()` — identical weights and edge count, random wiring. This is the control
used to test whether the *real* wiring matters for the task.

In [ ]:
s = g.shuffled(0).weights[np.ix_(order, order)]
fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
for ax, m, t in zip(axes, [w, s], ["MaleCNS wiring", "shuffled"]):
    ax.imshow(np.sign(m) * np.log1p(np.abs(m)) / np.log1p(lim), cmap="RdBu_r", vmin=-1, vmax=1); ax.set_title(t); ax.set_xticks([]); ax.set_yticks([])
plt.show()